In [40]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras

from scikeras.wrappers import KerasRegressor
from tensorflow.keras import backend as k
from tensorflow.keras.models import Sequential
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

In [41]:
# caminho para o csv
path = 'kc_house_data.csv'

# lendo o csv
df = pd.read_csv(path)
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,20140521T000000,360000.0,3,2.50,1530,1131,3.0,0,0,...,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,6600060120,20150223T000000,400000.0,4,2.50,2310,5813,2.0,0,0,...,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,1523300141,20140623T000000,402101.0,2,0.75,1020,1350,2.0,0,0,...,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,291310100,20150116T000000,400000.0,3,2.50,1600,2388,2.0,0,0,...,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


In [42]:
# tirando colunas que eu julgo desnecessárias
df.drop('id', axis=1, inplace=True)
df.drop('date', axis=1, inplace=True)
df

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,360000.0,3,2.50,1530,1131,3.0,0,0,3,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,400000.0,4,2.50,2310,5813,2.0,0,0,3,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,402101.0,2,0.75,1020,1350,2.0,0,0,3,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,400000.0,3,2.50,1600,2388,2.0,0,0,3,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


In [43]:
# vendo se há valores nulos
df.isnull().sum()

price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

In [44]:
# separando o X do Y
X = df.iloc[:, 1:19]
y = df.iloc[:, 0]

In [45]:
# fazendo a transformação dos valores
# O zipcode irá virar varias colunas (tratado como valor categorico)
# para os outros valores, eles serão todos padronizados
OneHotEncoder = ColumnTransformer(transformers=[('OneHot', OneHotEncoder(), ['zipcode']),], remainder=MinMaxScaler())
X = OneHotEncoder.fit_transform(X)

In [46]:
# vendo os formatos
X.shape, y.shape

((21613, 87), (21613,))

In [47]:
# função para criar a rede neural
def create_network():
    k.clear_session()
    # rede neural
    neural_network = Sequential([
        tf.keras.layers.InputLayer(shape=(87,)),
        tf.keras.layers.Dense(units=44, activation='relu'),
        tf.keras.layers.Dense(units=44, activation='relu'),
        tf.keras.layers.Dense(units=1, activation='linear'),
    ])
    # compilação
    neural_network.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mean_absolute_error'])
    return neural_network

In [48]:
# fazendo um treinamento simples
neural_network = create_network()
neural_network.fit(X, y, batch_size=50, epochs = 100)

Epoch 1/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 539447.9375 - mean_absolute_error: 539447.9375
Epoch 2/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 529757.4375 - mean_absolute_error: 529757.4375
Epoch 3/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 496569.5938 - mean_absolute_error: 496569.5938
Epoch 4/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 430166.7500 - mean_absolute_error: 430166.7500
Epoch 5/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 327109.4375 - mean_absolute_error: 327109.4062
Epoch 6/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 228131.7188 - mean_absolute_error: 228131.7188
Epoch 7/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 189708.4844 - mean_absolute_error: 189708.4844
Epoch 8/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 179624.3906 - mean_absolute_error: 179624.3906
Epoch 9/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 174923.4531 - mean_absolute_error: 174923.4531
Epoch 10/100
433/433 ━━━━━━━

In [49]:
# criando o modelo com o keras
neural_network = KerasRegressor(model=create_network, epochs=100, batch_size=50)

In [50]:
# fazendo o treinamento
results = cross_val_score(estimator=neural_network, X=X, y=y, cv=5, scoring='neg_mean_absolute_error')

Epoch 1/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 540052.3750 - mean_absolute_error: 540052.3750
Epoch 2/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 533857.1875 - mean_absolute_error: 533857.1875
Epoch 3/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 511794.0625 - mean_absolute_error: 511794.0625
Epoch 4/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 466078.7812 - mean_absolute_error: 466078.7812
Epoch 5/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391821.1562 - mean_absolute_error: 391821.1562
Epoch 6/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 292350.3125 - mean_absolute_error: 292350.3438
Epoch 7/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 215707.6406 - mean_absolute_error: 215707.6406
Epoch 8/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 187692.1406 - mean_absolute_error: 187692.1406
Epoch 9/100
346/346 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 179291.4531 - mean_absolute_error: 179291.4531
Epoch 10/100
346/346 ━━━━━━━